<a href="https://colab.research.google.com/github/daniarcear/diplomado-ia-udd-2026-arce/blob/main/Clase_24_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Clase 24 — Bonus track: cómo funciona RAG por dentro

**Diplomado IA Aplicada al Diseño · UDD · 2026** · Prof. Darío Osorio

Este es el **último notebook de Colab del curso**. Es opcional pero recomendado. Muestra en 20 minutos cómo funciona por dentro el patrón que usa NotebookLM (y muchas otras herramientas).

Después de esta clase pasamos 100% a herramientas SaaS.

## Objetivo
Construir un mini-RAG con 3 documentos ficticios y hacerle preguntas.

In [1]:
!pip install -q sentence-transformers numpy
print("Listo.")

Listo.


## Paso 1 — Nuestros "documentos"

En NotebookLM cargas PDFs. Acá simulamos con strings.

In [9]:
documentos = [
    """i. DESCRIPCIÓN GENERAL Nuestra estructura organizacional corresponde a la de un café de especialidad con formato de
restaurante, ubicado en una exclusiva zona comercial de Santiago de Chile. Nuestra propuesta de
valor se enfoca en la innovación de bebidas de café de alta calidad y en la minimización de
residuos mediante un modelo de economía circular. Un pilar fundamental es el diseño del espacio,
un local de dimensiones reducidas que potencia la exclusividad del café, complementado con una
propuesta de interiorismo y ornamentación que invitan a la conversación.""",

    """OBJETIVOS
1. Posicionarse como una cafetería de especialidad referente en el mercado nacional.
2. Destacar por su economía circular, donde el equipo de Investigación y desarrollo (i+d) es fuente
de nuevos productos fuera de la línea operativa clásica y se introducen al mercados tales como
cosmética y uso agrícola a partir de la reutilización de los granos molidos de café.
3. Innovar en la preparación de bebidas de café de especialidad funcionando hierbas nacionales
autóctonas
4.Capacitar el capital humano para consolidar una cultura de excelencia y calidad.""",

    """ACTORES/ROLES DE LA ORGANIZACIÓN
ACTOR ROL TIPO DE PARTICIPACIÓN INCUMBENCIA/INTERÉS
1. Marketing Manager Gestión del Marketing del
negocio (online u otros canales).
Revisión de research, encuestas
de satisfacción, etc.

Revisa las promoción de los
nuevos productos en manos de
agencia de publicidad/mktg y
community manager.

2.Administrador de Local Planificación, supervisión
operativa y financiera del café.
Control de stock, gestión de
proveedores, cumplimiento de
normativas y liderazgo del
personal para asegurar la
rentabilidad.

Gestión del negocio, optimiza
recursos y cumple metas
comerciales.

3.Head Barista Cargo enfocado en el liderazgo
técnico del equipo, control de
calidad, y capacitación de
nuevos baristas

Lidera y acompaña al equipo de
baristas, asegurando que cada
preparación de café cumpla con
los estándares de calidad de la
cafetería

4.Líder de I+D Investigación y creación de
nuevas bebidas y desarrollo de
subproductos a partir del residuo
del café para cosmética y
agricultura.

Analiza los datos de mercado
para materializar las ideas.
Realiza pruebas en la realización
de nuevos productos ecológicos
en base al café y finalmente
diseña nuevos modelos de
productos..""",
]
print(f"{len(documentos)} documentos cargados.")

3 documentos cargados.


## Paso 2 — Convertir documentos a embeddings

In [10]:
from sentence_transformers import SentenceTransformer
import numpy as np

modelo = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
embeddings = modelo.encode(documentos)
print(f"Shape: {embeddings.shape}")
print("Cada documento es ahora un vector de 384 dimensiones.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Shape: (3, 384)
Cada documento es ahora un vector de 384 dimensiones.


## Paso 3 — Función de búsqueda semántica

Le damos una pregunta, ella busca el documento más parecido.

In [16]:
def buscar(pregunta, top_k=1):
    p_emb = modelo.encode([pregunta])
    sims = np.dot(embeddings, p_emb.T).flatten()
    sims = sims / (np.linalg.norm(embeddings, axis=1) * np.linalg.norm(p_emb))
    top_idx = np.argsort(sims)[::-1][:top_k]
    return [(documentos[i], float(sims[i])) for i in top_idx]

preguntas = [
    "cuantos objetivos tenemos?",
    "cuantos actores hay?",
    "cual es el rol clave?",
]

for q in preguntas:
    print(f"P: {q}")
    for doc, sim in buscar(q, top_k=1):
        print(f"  Similitud: {sim:.2f}")
        print(f"  Documento: {doc[:200]}...")
    print()

P: cuantos objetivos tenemos?
  Similitud: 0.37
  Documento: OBJETIVOS
1. Posicionarse como una cafetería de especialidad referente en el mercado nacional.
2. Destacar por su economía circular, donde el equipo de Investigación y desarrollo (i+d) es fuente
de nu...

P: cuantos actores hay?
  Similitud: 0.33
  Documento: ACTORES/ROLES DE LA ORGANIZACIÓN
ACTOR ROL TIPO DE PARTICIPACIÓN INCUMBENCIA/INTERÉS
1. Marketing Manager Gestión del Marketing del
negocio (online u otros canales).
Revisión de research, encuestas
de...

P: cual es el rol clave?
  Similitud: 0.39
  Documento: i. DESCRIPCIÓN GENERAL Nuestra estructura organizacional corresponde a la de un café de especialidad con formato de
restaurante, ubicado en una exclusiva zona comercial de Santiago de Chile. Nuestra p...



## Cierre

Esto es exactamente lo que hace NotebookLM (más pulido y con Gemini razonando encima). Ahora que viste el mecanismo, ya sabes por qué:

- Cuando la respuesta "no está en las fuentes", el sistema puede decir "no sé".
- Las citas son literales (vienen del documento recuperado).
- La calidad depende del corte (chunking) y de qué tan buenas son tus fuentes.

**No hace falta que ejecutes esto en tu proyecto** — usa NotebookLM directamente. Este notebook es solo para entender qué pasa por debajo.